# SAC Collector Head-to-Head — Supplement: ReBRAC β1=1.0 × m_multi_mix × 3 seed

**目标**：补 FQL P2 v1.4 `m_multi_mix_50priv_50goal_1000` cell 缺的 **ReBRAC β1=1.0** sweep,让 sprint 1+2 + FQL P2 cross-source matrix 完整。

**Why this cell**:
- FQL P2 v1.4 m_multi_mix 已有: ReBRAC (β1=4.0 default) seed [42, 0] μ=0.990 + FQL seed [42, 0] μ=0.955 → Δ=−0.035 GRAY (CI 跨 0)
- **缺**: ReBRAC β1=1.0 m_multi_mix(spec line 448 `medium ↔ m_multi_mix | (N/A: 仓库无 β1=1 sweep on m_multi_mix)`)
- Sprint 1+2 SAC matrix 上 ReBRAC β1=1 strictly dominate β1=4 on mexp(CI 不跨 0); m_multi_mix 上是否 reproduce 是 cross-source mechanism universality 的 key cell

**前置 commits (已闭环)**:
- `97d394c` — SACCheckpointPolicy adapter (sprint 1+2 用同一 adapter)
- `104c14c` — sprint 1 ReBRAC × SAC 4 tier × 3 seed completed (24 run)
- `fc5cbce` — sprint 2 FQL × SAC 4 tier × 3 seed completed (12 run)
- FQL P2 v1.4 (`docs/fql_succession_p2_main_spec.md` v1.4): m_multi_mix β1=4 + FQL 已存

**配置** (mirror FQL P2 v1.4 m_multi_mix run notebook protocol, only β1 changed):

| 项 | 值 | 来源 |
|---|---|---|
| Algo | ReBRAC (vanilla critic, critic_LN=on) | FQL P2 v1.4 m_multi_mix run notebook |
| β1 / β2 | **1.0 / 2.0** | sprint 1 finding (β1=1 dominate β1=4 on mexp CI 不跨 0) |
| Dataset | `offline_data/fql_succession/m_multi_mix_50priv_50goal_1000` (1000 ep, 50% privileged + 50% goalseek mix) | FQL P2 v1.4 |
| Seeds | [42, 0, 7] | seed [42, 0] paired with FQL P2 v1.4; seed 7 extends for own bootstrap n_seeds=3 |
| Total steps | 200_000 (uniform sampling) | FQL P2 v1.4 |
| batch / hidden / layers | 256 / 256 / 3 | FQL P2 v1.4 |
| lr / γ / τ | 3e-4 / 0.99 / 0.005 | FQL P2 v1.4 |
| Manifest (eval) | `benchmarks/single_u10_cross_tgt15_ep100.json` (100 ep) | FQL P2 v1.4 (注意:与 sprint 1+2 SAC 矩阵的 30-ep manifest 不同,这是 cross-source paired 必须) |
| test_seed | 456 | FQL P2 v1.4 evaluate_offline default |

**预算**:

| 阶段 | run | wallclock |
|---|---:|---:|
| Train: ReBRAC β1=1.0 × m_multi_mix × 3 seed | 3 | ~90 min L4 |
| evaluate_offline 100-ep × 3 run | 3 | ~10 min L4 |
| **Supplement total** | **3** | **~100 min L4** |

→ 1 个 Colab Pro L4 session 紧凑可完(单 session, 不拆 seed)。

**Pre-registered hypothesis (per spec §4.0.10 待落)**:
- **H1 (paired vs FQL P2 v1.4 ReBRAC β1=4 m_multi_mix)**: ReBRAC β1=1.0 m_multi_mix ≥ ReBRAC β1=4.0 m_multi_mix
  - 复现 sprint 1 mexp finding (β1=1 wins) on cross-source → universal across data source
  - 否则 (β1=4 wins or both saturate) → SAC source idiosyncrasy or saturated regime
- **H2 (paired vs FQL P2 v1.4 FQL m_multi_mix)**: ReBRAC β1=1.0 m_multi_mix vs FQL m_multi_mix direction?
  - 若 ReBRAC β1=1 ≥ FQL → 复现 FQL P2 v1.4 finding (β1=1 universal weak dominance over FQL)
  - 若 ReBRAC β1=1 < FQL → reverse of FQL P2 finding (FQL outperforms when properly tuned ReBRAC)

**Skip-resume**:`agent_final.pt` 存在则 skip 该 seed(与 sprint 1+2 同协议)。

**实时输出**:`!python ...` IPython shell magic(per memory feedback_notebook_realtime_output)。

## 0. 环境检查

In [1]:
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name:    {torch.cuda.get_device_name(0)}")

PyTorch:        2.10.0+cu128
CUDA available: True
Device name:    NVIDIA L4


## 1. 挂载 Drive + cd 到项目根

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd drive/"MyDrive"/"Colab Notebooks"/"new_offRL"/"rl_v2_5"
!pwd

/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 2. Sanity check — dataset + manifest + flow 存在

**预期**:3 项 ✅。任何 ⚠️ 都需停下定位(不要跳过 sanity 直接开跑 1.5h training)。

In [4]:
from pathlib import Path
import json

DATASET_DIR  = Path("offline_data/fql_succession/m_multi_mix_50priv_50goal_1000")
TRANSITIONS  = DATASET_DIR / "transitions.npz"
METADATA     = DATASET_DIR / "metadata.json"
MANIFEST     = Path("benchmarks/single_u10_cross_tgt15_ep100.json")
FLOW         = Path("wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy")

all_ok = True
for p in (TRANSITIONS, MANIFEST, FLOW):
    if p.exists():
        print(f"  ✅ {p}  ({p.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"  ⚠️ {p}  NOT FOUND")
        all_ok = False

if METADATA.exists():
    meta = json.loads(METADATA.read_text())
    print(f"\n[metadata] {METADATA}")
    for k in ("policy_mixture", "action_noise_std", "num_episodes", "num_transitions",
              "probe_layout", "task_geometry", "objective", "mean_return", "success_rate"):
        if k in meta:
            v = meta[k]
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
else:
    print(f"  ⚠️ metadata.json NOT FOUND at {METADATA}")
    all_ok = False

print()
if all_ok:
    print("✅ Sanity check passed — 可以开始 training.")
else:
    raise FileNotFoundError("Sanity check failed — 见上面 ⚠️ 项,定位后 rerun.")

  ✅ offline_data/fql_succession/m_multi_mix_50priv_50goal_1000/transitions.npz  (16.5 MB)
  ✅ benchmarks/single_u10_cross_tgt15_ep100.json  (0.1 MB)
  ✅ wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy  (230.4 MB)

[metadata] offline_data/fql_succession/m_multi_mix_50priv_50goal_1000/metadata.json
  policy_mixture: [{'policy': 'privileged', 'weight': 1.0}]
  action_noise_std: 0.1000
  num_episodes: 1000
  num_transitions: 138290
  probe_layout: s0
  task_geometry: cross_stream
  objective: arrival_v2
  mean_return: 48.2727
  success_rate: 0.8260

✅ Sanity check passed — 可以开始 training.


## 3. Config + skip-resume plan

**Run identifier**:`rebrac_b1_1p0__m_multi_mix__seed_{42|0|7}`

**Skip-resume**:检查 `<save-dir>/agent_final.pt` 存在则 skip(train 结束才写,不会 false-positive)。

In [5]:
SEEDS = [42, 0, 7]
BETA_TAG = "b1_1p0"
ACTOR_PEN = 1.0
CRITIC_PEN = 2.0
CELL_ID = "m_multi_mix"

CKPT_ROOT    = Path("checkpoints/offline/sac_collector_h2h/xsource_supplement") / BETA_TAG / CELL_ID
RESULTS_ROOT = Path("results/offline/sac_collector_h2h/xsource_supplement") / BETA_TAG / CELL_ID

def save_dir(seed: int) -> Path:
    return CKPT_ROOT / f"seed_{seed}"

def test_result_path(seed: int) -> Path:
    return RESULTS_ROOT / f"seed_{seed}" / "test_result.json"

n_total = 0
n_done  = 0
for seed in SEEDS:
    n_total += 1
    sd = save_dir(seed)
    done = (sd / "agent_final.pt").exists()
    if done:
        n_done += 1
    status = "✅ DONE" if done else "⏳ TODO"
    print(f"  [{status}]  {BETA_TAG} | {CELL_ID} | seed={seed}  →  {sd}")

print(f"\n=== {n_done}/{n_total} runs already complete (skip-resume) ===")

  [⏳ TODO]  b1_1p0 | m_multi_mix | seed=42  →  checkpoints/offline/sac_collector_h2h/xsource_supplement/b1_1p0/m_multi_mix/seed_42
  [⏳ TODO]  b1_1p0 | m_multi_mix | seed=0  →  checkpoints/offline/sac_collector_h2h/xsource_supplement/b1_1p0/m_multi_mix/seed_0
  [⏳ TODO]  b1_1p0 | m_multi_mix | seed=7  →  checkpoints/offline/sac_collector_h2h/xsource_supplement/b1_1p0/m_multi_mix/seed_7

=== 0/3 runs already complete (skip-resume) ===


## 4. Train — ReBRAC β1=1.0 × m_multi_mix × 3 seed (3 run, ~90 min L4)

**协议** (mirror FQL P2 v1.4 m_multi_mix ReBRAC, β1=1.0 instead of 4.0,其它一致):
- `--actor-penalty-coef 1.0 --critic-penalty-coef 2.0`(β2 锁 2.0)
- `--critic-layernorm --no-actor-layernorm`(per FQL P2 broad val v2 N0 anchor)
- 200k step, batch 256, hidden 256, 3 layers, lr=3e-4, γ=0.99, τ=0.005
- `--sampling-mode uniform`(no PER)
- `--skip-final-eval`(独立 eval cell 用 evaluate_offline)

**实时输出**:`!python ...` IPython shell magic。

In [6]:
import time

t_start = time.time()

npz = str(TRANSITIONS)
manifest_str = str(MANIFEST)

for seed in SEEDS:
    sd = save_dir(seed)
    sd_str = str(sd)
    if (sd / "agent_final.pt").exists():
        print(f"[skip-train] {BETA_TAG} | {CELL_ID} | seed={seed} (agent_final.pt exists)")
        continue
    sd.mkdir(parents=True, exist_ok=True)
    print(f"\n{'=' * 72}")
    print(f"  train {BETA_TAG} | {CELL_ID} | seed={seed}  →  {sd}")
    print(f"{'=' * 72}")
    t0 = time.time()
    !python -m scripts.train_offline \
        --algo rebrac \
        --offline-data '{npz}' \
        --manifest '{manifest_str}' \
        --probe-layout s0 \
        --history-length 4 \
        --task-geometry cross_stream \
        --target-speed 1.5 \
        --objective arrival_v2 \
        --sampling-mode uniform \
        --total-steps 200000 \
        --batch-size 256 \
        --hidden-dim 256 \
        --num-hidden-layers 3 \
        --actor-lr 3e-4 \
        --critic-lr 3e-4 \
        --gamma 0.99 \
        --tau 0.005 \
        --actor-penalty-coef {ACTOR_PEN} \
        --critic-penalty-coef {CRITIC_PEN} \
        --policy-noise 0.2 \
        --noise-clip 0.5 \
        --policy-freq 2 \
        --grad-clip-norm 10.0 \
        --normalizer-eps 1e-3 \
        --critic-layernorm \
        --no-actor-layernorm \
        --eval-every 0 \
        --skip-final-eval \
        --log-every 1000 \
        --seed {seed} \
        --device cuda \
        --save-dir '{sd_str}'
    print(f"[train done] {BETA_TAG} | {CELL_ID} | seed={seed} in {(time.time() - t0) / 60:.1f} min")

print(f"\n=== Train all complete, total = {(time.time() - t_start) / 60:.1f} min ===")


  train b1_1p0 | m_multi_mix | seed=42  →  checkpoints/offline/sac_collector_h2h/xsource_supplement/b1_1p0/m_multi_mix/seed_42
[offline] algo=rebrac transitions=138290 obs_dim=48 action_dim=2 protocol=deployable tensor_replay=on eval_workers=1 sampling=uniform total_steps=200000
[train] step=1 q=-0.858 critic=45.923 actor=2.058 bc=1.058 lambda=1.165 critic_pen=1.136
[train] step=1000 q=-2.799 critic=1.396 actor=0.709 bc=0.076 lambda=0.226 critic_pen=0.172
[train] step=2000 q=-1.602 critic=2.957 actor=0.300 bc=0.086 lambda=0.134 critic_pen=0.146
[train] step=3000 q=0.220 critic=165.807 actor=0.092 bc=0.112 lambda=0.091 critic_pen=0.158
[train] step=4000 q=-1.143 critic=3.550 actor=0.168 bc=0.069 lambda=0.087 critic_pen=0.135
[train] step=5000 q=-3.283 critic=205.989 actor=0.315 bc=0.078 lambda=0.072 critic_pen=0.186
[train] step=6000 q=-0.530 critic=184.998 actor=0.084 bc=0.051 lambda=0.062 critic_pen=0.130
[train] step=7000 q=-2.886 critic=1.621 actor=0.232 bc=0.062 lambda=0.059 criti

## 5. Eval — 3 run × 100 ep test on `single_u10_cross_tgt15_ep100`

**Per-run skip-resume**:`test_result.json` 存在则 skip。

**预算**:3 × ~3 min = ~10 min L4。

In [7]:
import time

t_start = time.time()

for seed in SEEDS:
    sd = save_dir(seed)
    sd_str = str(sd)
    out_path = test_result_path(seed)
    out_path_str = str(out_path)
    manifest_str = str(MANIFEST)
    if out_path.exists():
        print(f"[skip-eval] {BETA_TAG} | {CELL_ID} | seed={seed} (test_result.json exists)")
        continue
    if not (sd / "agent_final.pt").exists():
        print(f"[warn] {BETA_TAG} | {CELL_ID} | seed={seed}  missing agent_final.pt — train not done?")
        continue
    out_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n{'=' * 72}")
    print(f"  eval  {BETA_TAG} | {CELL_ID} | seed={seed}  →  {out_path}")
    print(f"{'=' * 72}")
    t0 = time.time()
    !python -m scripts.evaluate_offline \
        --checkpoint '{sd_str}' \
        --manifest '{manifest_str}' \
        --episodes 100 \
        --seed 456 \
        --device cuda \
        --output-json '{out_path_str}'
    print(f"[eval done] {BETA_TAG} | {CELL_ID} | seed={seed} in {(time.time() - t0) / 60:.1f} min")

print(f"\n=== Eval all complete, total = {(time.time() - t_start) / 60:.1f} min ===")


  eval  b1_1p0 | m_multi_mix | seed=42  →  results/offline/sac_collector_h2h/xsource_supplement/b1_1p0/m_multi_mix/seed_42/test_result.json
reward_objective      : arrival_v2
energy_cost_gain      : 0.000000
safety_cost_gain      : 0.000000
episodes              : 100
success_rate          : 99.0%
avg_return            : 137.80 +/- 32.76
avg_safety_cost       : 1.802 +/- 1.602
avg_time_s            : 41.51 +/- 6.39
avg_time_s_success    : 41.48
avg_energy            : 32566.94 +/- 5175.32
avg_path_length_m     : 45.00 +/- 5.37
avg_progress_ratio    : 0.905 +/- 0.024
avg_path_efficiency   : 0.871 +/- 0.087
termination           : {'goal': 99, 'out_of_bounds': 1}
benchmark_manifest    : benchmarks/single_u10_cross_tgt15_ep100.json
[eval done] b1_1p0 | m_multi_mix | seed=42 in 1.8 min

  eval  b1_1p0 | m_multi_mix | seed=0  →  results/offline/sac_collector_h2h/xsource_supplement/b1_1p0/m_multi_mix/seed_0/test_result.json
reward_objective      : arrival_v2
energy_cost_gain      : 0.000000

## 6. Inline summary — 3 test_result.json → table + paired vs FQL P2 v1.4

**输出**:
- per-seed success_rate + return
- 3-seed mean ± std
- Paired vs FQL P2 v1.4 ReBRAC β1=4 m_multi_mix (seed [42, 0] only paired; seed 7 informational)
- Paired vs FQL P2 v1.4 FQL m_multi_mix (seed [42, 0] only paired)

详细 stratified bootstrap CI 留给本地 `scripts/analyze_sac_collector_h2h_joint.py` extension。

In [8]:
import json
import numpy as np

missing = []
rows = []
for seed in SEEDS:
    p = test_result_path(seed)
    if not p.exists():
        missing.append(seed)
        continue
    d = json.loads(p.read_text())
    rows.append({
        "seed": seed,
        "sr":   float(d.get("eval_success_rate", float("nan"))),
        "ret":  float(d.get("eval_return", float("nan"))),
        "n":    int(d.get("num_eval_episodes", 0)),
    })

print("=== ReBRAC β1=1.0 × m_multi_mix supplement (test 100 ep × 3 seed) ===\n")
print(f"{'seed':>6s}  {'n_ep':>5s}  {'success_rate':>14s}  {'mean_return':>14s}")
print("-" * 50)
for r in rows:
    print(f"{r['seed']:>6d}  {r['n']:>5d}  {r['sr']:>14.4f}  {r['ret']:>14.4f}")

if rows:
    sr_all = np.array([r["sr"] for r in rows])
    print("-" * 50)
    print(f"  μ ± σ (n={len(sr_all)} seeds):  {sr_all.mean():.4f} ± {sr_all.std(ddof=1) if len(sr_all)>1 else 0:.4f}")

if missing:
    print(f"\n⚠️ {len(missing)} runs missing test_result.json: seeds={missing}")
else:
    print("\n✅ All 3 runs have test_result.json — supplement complete.")

# ---- Paired vs FQL P2 v1.4 (from docs/assets/fql_succession_p2/test_eval_summary.md) ----
FQLP2_M_MULTI_MIX = {
    "rebrac_b1_4": {42: 1.000, 0: 0.980},   # μ=0.990
    "fql":         {42: 0.960, 0: 0.950},   # μ=0.955
}

if rows and not missing:
    print("\n=== Paired vs FQL P2 v1.4 m_multi_mix (seed-matched [42, 0] only) ===\n")
    own_by_seed = {r["seed"]: r["sr"] for r in rows}
    for ref_name, ref_data in FQLP2_M_MULTI_MIX.items():
        deltas = []
        for s in (42, 0):
            if s in own_by_seed and s in ref_data:
                d = own_by_seed[s] - ref_data[s]
                deltas.append(d)
                print(f"  seed={s}: β1=1 supplement {own_by_seed[s]:.3f}  vs  FQL P2 {ref_name} {ref_data[s]:.3f}  →  Δ {d:+.3f}")
        if deltas:
            print(f"  paired μ Δ (own − FQL P2 {ref_name}): {np.mean(deltas):+.4f}  (n_paired={len(deltas)} seed)")
        print()

=== ReBRAC β1=1.0 × m_multi_mix supplement (test 100 ep × 3 seed) ===

  seed   n_ep    success_rate     mean_return
--------------------------------------------------
    42    100          0.9900        137.7974
     0    100          0.9900        137.2130
     7    100          0.9800        133.6368
--------------------------------------------------
  μ ± σ (n=3 seeds):  0.9867 ± 0.0058

✅ All 3 runs have test_result.json — supplement complete.

=== Paired vs FQL P2 v1.4 m_multi_mix (seed-matched [42, 0] only) ===

  seed=42: β1=1 supplement 0.990  vs  FQL P2 rebrac_b1_4 1.000  →  Δ -0.010
  seed=0: β1=1 supplement 0.990  vs  FQL P2 rebrac_b1_4 0.980  →  Δ +0.010
  paired μ Δ (own − FQL P2 rebrac_b1_4): +0.0000  (n_paired=2 seed)

  seed=42: β1=1 supplement 0.990  vs  FQL P2 fql 0.960  →  Δ +0.030
  seed=0: β1=1 supplement 0.990  vs  FQL P2 fql 0.950  →  Δ +0.040
  paired μ Δ (own − FQL P2 fql): +0.0350  (n_paired=2 seed)



## 7. Next steps (after supplement closes)

1. **回收 _completed.ipynb 副本** commit 进 git 作实验记录(参考 audit / collection / sprint 1+2 completed 模式)。
2. **本地 joint analysis**:把本 cell 3 个 `test_result.json` 落到 `scripts/analyze_sac_collector_h2h_joint.py` extension(加一组 cross-source contrasts),重新跑 stratified bootstrap:
   - Contrast 1 (own SAC matrix): β1=1 m_multi_mix vs sprint 1 β1=1 mexp / β1=1 expert (direction universality)
   - Contrast 2 (paired vs FQL P2 v1.4): β1=1 m_multi_mix vs β1=4 m_multi_mix (replicate sprint 1 mexp finding)
   - Contrast 3 (paired vs FQL P2 v1.4): β1=1 m_multi_mix vs FQL m_multi_mix (cross-source mechanism universality)
3. **Land verdict to spec §4.0.10 + summary §4.4**:cross-source contradiction narrative + paper closure direction。
4. **(Optional)** 如果 m_multi_mix β1=1 在 paired bootstrap 显著超 β1=4 → confirms sprint 1 mexp finding cross-source。若 β1=1 ≤ β1=4 → SAC source idiosyncrasy or m_multi_mix saturation.